In [1]:
import numpy as np
import pandas as pd
import os
import sys
import glob
import tqdm
import re
from typing import List

sys.path.append("..")
from src import text_extraction, create_sentence_nace_code_similarities, analysis_functions
import test_base
from sentence_splitter import split_text_into_sentences

## Test retrieving the similarities for chunks in a pdf to the NACE Code

**Function:** pdf-> (chunk x code -> [-1,1])

**Parameters:** 

- pdf_path
- way of chunking the text (e.g. sentences, sliding window, or paragraphs)
- way of preprocessing (most is fixed for all reports)
    - similarity threshold of relevant chunks
    - length of irrelevant chunks

**Store analytics for each datapoint:**

- mean score for each class given a threshold

In [2]:
# Parameters: 
cos_threshold = 0.0

In [3]:
dataset_path = "../data/german_annual_reports"
dataset_path = "../data/PDF_stoxx600"
dataset_path_texts = "../data/TEXT_stoxx600_docling"

In [4]:
nace_classes = pd.read_excel(os.path.join(dataset_path, "STOXX600_as_of_2025_03_13.xlsx"))
nace_classes.head()

,Name,Symbol,FactSet ID,Revenue - 2022 (in EUR),Revenue - 2023 (in EUR),Revenue - 2024 (in EUR),NACE,NACE_letter,Report
0,Brenntag SE,BNR-DE,BNR-DE,19429.3000,16815.100000,NaN,46.73,G,Brenntag Societas Europaea1.pdf
1,Grafton Group Plc,GFTU-GB,GFTU-GB,2697.4207,2666.632525,NaN,47.52,G,Grafton Group Plc1.pdf
2,LANXESS AG,LXS-DE,LXS-DE,8088.0000,6714.000000,NaN,20.59,C,LANXESS AG1.pdf
3,Banco de Sabadell SA,SAB-ES,SAB-ES,6927.9200,10467.899000,11656.548,64.19,K,Banco de Sabadell SA1.pdf
4,freenet AG,FNTN-DE,FNTN-DE,2556.7140,2627.300000,NaN,61.20,J,freenet AG1.pdf


In [5]:
report_to_nace_class = nace_classes.dropna(subset=["Report"]).set_index('Report').to_dict()["NACE"]
report_to_nace_class

{'Brenntag Societas Europaea1.pdf': 46.73,
 'Grafton Group Plc1.pdf': 47.52,
 'LANXESS AG1.pdf': 20.59,
 'Banco de Sabadell SA1.pdf': 64.19,
 'freenet AG1.pdf': 61.2,
 'Gerresheimer AG1.pdf': 22.22,
 'TUI AG1.pdf': 79.12,
 'InterContinental Hotels Group PLC1.pdf': 55.1,
 'Verallia SAS3.pdf': 23.14,
 'LEG Immobilien SE1.pdf': 68.32,
 'Raiffeisen Bank International AG3.pdf': 64.19,
 'ITV plc1.pdf': 59.11,
 'Helvetia Holding Ltd2.pdf': 65.11,
 'Telecom Italia S.p.A.3.pdf': 61.2,
 'Euronext NV1.pdf': 66.11,
 'BELIMO Holding AG1.pdf': 28.12,
 'SalMar ASA1.pdf': 3.21,
 'Antofagasta plc1.pdf': 7.29,
 'Ashtead Group plc1.pdf': 77.32,
 'Severn Trent Plc1.pdf': 36.0,
 'Bellway p.l.c.1.pdf': 41.2,
 'Balfour Beatty plc1.pdf': 42.11,
 'British Land Company PLC1.pdf': 68.2,
 'Bunzl plc1.pdf': 46.75,
 'Vistry Group PLC3.pdf': 41.2,
 'Aviva plc1.pdf': 65.11,
 'Diageo PLC1.pdf': 11.01,
 'DCC Plc1.pdf': 35.22,
 'BAE Systems plc1.pdf': 30.3,
 'Derwent London PLC REIT1.pdf': 68.2,
 'Big Yellow Group PLC1.

In [6]:
report_to_nace_class = {report[0][:-4] + ".txt": report[1] for report in report_to_nace_class.items()}
report_to_nace_class

{'Brenntag Societas Europaea1.txt': 46.73,
 'Grafton Group Plc1.txt': 47.52,
 'LANXESS AG1.txt': 20.59,
 'Banco de Sabadell SA1.txt': 64.19,
 'freenet AG1.txt': 61.2,
 'Gerresheimer AG1.txt': 22.22,
 'TUI AG1.txt': 79.12,
 'InterContinental Hotels Group PLC1.txt': 55.1,
 'Verallia SAS3.txt': 23.14,
 'LEG Immobilien SE1.txt': 68.32,
 'Raiffeisen Bank International AG3.txt': 64.19,
 'ITV plc1.txt': 59.11,
 'Helvetia Holding Ltd2.txt': 65.11,
 'Telecom Italia S.p.A.3.txt': 61.2,
 'Euronext NV1.txt': 66.11,
 'BELIMO Holding AG1.txt': 28.12,
 'SalMar ASA1.txt': 3.21,
 'Antofagasta plc1.txt': 7.29,
 'Ashtead Group plc1.txt': 77.32,
 'Severn Trent Plc1.txt': 36.0,
 'Bellway p.l.c.1.txt': 41.2,
 'Balfour Beatty plc1.txt': 42.11,
 'British Land Company PLC1.txt': 68.2,
 'Bunzl plc1.txt': 46.75,
 'Vistry Group PLC3.txt': 41.2,
 'Aviva plc1.txt': 65.11,
 'Diageo PLC1.txt': 11.01,
 'DCC Plc1.txt': 35.22,
 'BAE Systems plc1.txt': 30.3,
 'Derwent London PLC REIT1.txt': 68.2,
 'Big Yellow Group PLC1.

In [7]:
reports_path = glob.glob(os.path.join(dataset_path_texts, "*.txt"))
reports_path

['../data/TEXT_stoxx600_docling/Hannover Rueck SE1.txt',
 '../data/TEXT_stoxx600_docling/Interpump Group S.p.A.1.txt',
 '../data/TEXT_stoxx600_docling/Intertek Group PLC1.txt',
 '../data/TEXT_stoxx600_docling/Anheuser-Busch InBev SANV3.txt',
 '../data/TEXT_stoxx600_docling/Scout24 SE3.txt',
 '../data/TEXT_stoxx600_docling/Bridgepoint Group Plc1.txt',
 '../data/TEXT_stoxx600_docling/Financiere de Tubize SA2.txt',
 '../data/TEXT_stoxx600_docling/Severn Trent Plc1.txt',
 '../data/TEXT_stoxx600_docling/Bakkafrost PF2.txt',
 '../data/TEXT_stoxx600_docling/Ferrari NV2.txt',
 '../data/TEXT_stoxx600_docling/Sika AG3.txt',
 '../data/TEXT_stoxx600_docling/Swiss Prime Site AG2.txt',
 '../data/TEXT_stoxx600_docling/Poste Italiane SpA2.txt',
 '../data/TEXT_stoxx600_docling/Haleon PLC1.txt',
 '../data/TEXT_stoxx600_docling/Land Securities Group PLC2.txt',
 '../data/TEXT_stoxx600_docling/Rentokil Initial plc2.txt',
 '../data/TEXT_stoxx600_docling/Nexans SA3.txt',
 '../data/TEXT_stoxx600_docling/SEB S

In [8]:
def preprocess_report_tables_only(pdf_path: str) -> List[str]:

    with open(pdf_path, "r") as f: 
        text = f.read()
        
    lines = text.split("\n")

    tables = []
    current_table = []

    for line in lines:
        if line.strip().startswith("|"):  # line belongs to a table
            current_table.append(line.strip())
        else:
            if current_table:  # table ended
                tables.append("\n".join(current_table))
                current_table = []

    # catch last table if file ends without empty lines
    if current_table:
        tables.append("\n".join(current_table))

    return tables

In [9]:
for i in range(2, 5): 
    nace_level = i

    result_path = f"../results/tables_cos_sim_{cos_threshold}_nace_level_{nace_level}_stoxx"

    res = test_base.test_similarities(reports_path, preprocess_report_tables_only, 0, cos_threshold, report_to_nace_class, result_path, level=i)

  0%|          | 0/258 [00:00<?, ?it/s]

Report:  ../data/TEXT_stoxx600_docling/Hannover Rueck SE1.txt
Report:  ../data/TEXT_stoxx600_docling/Interpump Group S.p.A.1.txt
Report:  ../data/TEXT_stoxx600_docling/Intertek Group PLC1.txt
Report:  ../data/TEXT_stoxx600_docling/Anheuser-Busch InBev SANV3.txt
Report:  ../data/TEXT_stoxx600_docling/Scout24 SE3.txt
Report:  ../data/TEXT_stoxx600_docling/Bridgepoint Group Plc1.txt
Report:  ../data/TEXT_stoxx600_docling/Financiere de Tubize SA2.txt
Report:  ../data/TEXT_stoxx600_docling/Severn Trent Plc1.txt
Report:  ../data/TEXT_stoxx600_docling/Bakkafrost PF2.txt
Report:  ../data/TEXT_stoxx600_docling/Ferrari NV2.txt
Report:  ../data/TEXT_stoxx600_docling/Sika AG3.txt
Report:  ../data/TEXT_stoxx600_docling/Swiss Prime Site AG2.txt
Report:  ../data/TEXT_stoxx600_docling/Poste Italiane SpA2.txt
Report:  ../data/TEXT_stoxx600_docling/Haleon PLC1.txt
Report:  ../data/TEXT_stoxx600_docling/Land Securities Group PLC2.txt
Report:  ../data/TEXT_stoxx600_docling/Rentokil Initial plc2.txt
Report

 14%|█▍        | 36/258 [00:05<00:36,  6.10it/s]

38
96
18
37
63
7
93
56
73
86
78
8
42
60
53
39
72
62
41
88
97
68
35
85
84
99
Report:  ../data/TEXT_stoxx600_docling/Alstom SA2.txt
Report:  ../data/TEXT_stoxx600_docling/QinetiQ Group plc1.txt
Report:  ../data/TEXT_stoxx600_docling/Rexel SA1.txt
Report:  ../data/TEXT_stoxx600_docling/COMET Holding AG1.txt
Report:  ../data/TEXT_stoxx600_docling/Yara International ASA2.txt
Report:  ../data/TEXT_stoxx600_docling/Telia Company AB2.txt
Report:  ../data/TEXT_stoxx600_docling/RELX PLC1.txt
Report:  ../data/TEXT_stoxx600_docling/ITV plc1.txt
Report:  ../data/TEXT_stoxx600_docling/TUI AG1.txt
Report:  ../data/TEXT_stoxx600_docling/Legal & General Group Plc1.txt
Report:  ../data/TEXT_stoxx600_docling/Wendel SE2.txt
Report:  ../data/TEXT_stoxx600_docling/Banca Popolare di Sondrio S.p.A.1.txt
Report:  ../data/TEXT_stoxx600_docling/Aviva plc1.txt
Report:  ../data/TEXT_stoxx600_docling/Ipsen SA2.txt
Report:  ../data/TEXT_stoxx600_docling/Stellantis N.V.1.txt
Report:  ../data/TEXT_stoxx600_docling/Uni

 21%|██▏       | 55/258 [00:08<00:32,  6.30it/s]

...

25.11
24.54
27.52
28.12
32.13
28.13
32.11
31.09
31.03
31.02
30.99
30.12
29.31
29.20
28.99
28.96
28.94
28.92
28.91
28.41
28.30
28.29
28.25
28.22
28.21
28.15
28.14
42.22
Report:  ../data/TEXT_stoxx600_docling/Flughafen Zurich AG1.txt
Report:  ../data/TEXT_stoxx600_docling/Akzo Nobel N.V.2.txt
Report:  ../data/TEXT_stoxx600_docling/Randstad NV1.txt
Report:  ../data/TEXT_stoxx600_docling/Barry Callebaut AG3.txt
Report:  ../data/TEXT_stoxx600_docling/ASR Nederland N.V.1.txt
Report:  ../data/TEXT_stoxx600_docling/Banque Cantonale Vaudoise1.txt
Report:  ../data/TEXT_stoxx600_docling/argenx SE1.txt
Report:  ../data/TEXT_stoxx600_docling/Wihlborgs Fastigheter AB2.txt
Report:  ../data/TEXT_stoxx600_docling/Swiss Life Holding AG3.txt
Report:  ../data/TEXT_stoxx600_docling/EQT AB1.txt
Number of Chunks:  1
66.3
80.30
66.21
46.19
25.21
47.54
45.31
45.32
47.74
47.77
47.81
47.82
47.24
46.15
46.11
14.12
74.30
11.06
47.42
47.76
46.13
46.18
46.23
01.14
46.16
46.24
46.32
46.17
46.63
46.35
46.48
46.37
46.3

 85%|████████▍ | 219/258 [34:37<02:07,  3.27s/it]

Report:  ../data/TEXT_stoxx600_docling/Halma plc1.txt
Report:  ../data/TEXT_stoxx600_docling/London Stock Exchange Group plc1.txt
Report:  ../data/TEXT_stoxx600_docling/Volkswagen AG1.txt
Report:  ../data/TEXT_stoxx600_docling/UNITE Group plc3.txt
Report:  ../data/TEXT_stoxx600_docling/Eurofins Scientific SE2.txt
Report:  ../data/TEXT_stoxx600_docling/TAG Immobilien AG3.txt
Report:  ../data/TEXT_stoxx600_docling/Porsche Automobil Holding SE1.txt
Report:  ../data/TEXT_stoxx600_docling/Daimler Truck Holding AG1.txt
Report:  ../data/TEXT_stoxx600_docling/Cembra Money Bank AG1.txt
Report:  ../data/TEXT_stoxx600_docling/Geberit AG1.txt
Report:  ../data/TEXT_stoxx600_docling/Bank of Ireland Group Plc1.txt
Report:  ../data/TEXT_stoxx600_docling/Smiths Group PLC1.txt
Report:  ../data/TEXT_stoxx600_docling/Fortnox AB1.txt
Report:  ../data/TEXT_stoxx600_docling/NatWest Group Plc3.txt
Report:  ../data/TEXT_stoxx600_docling/DSM-Firmenich AG1.txt
Report:  ../data/TEXT_stoxx600_docling/Kingspan Grou

100%|██████████| 258/258 [34:56<00:00,  8.13s/it]

20.14
95.25
26.60
96.02
24.54
13.91
10.62
80.30
13.20
02.30
95.11
24.46
75.00
95.22
95.12
25.50
10.61
86.90
01.25
14.20
26.70
28.41
87.10
95.21
74.20
72.11
Report:  ../data/TEXT_stoxx600_docling/Symrise AG2.txt
Report:  ../data/TEXT_stoxx600_docling/Phoenix Group Holdings plc1.txt
Report:  ../data/TEXT_stoxx600_docling/Bellway p.l.c.1.txt
